In [ ]:
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(usecwd=False)), "The .env file was not loaded."

from pathlib import Path
from dataclasses import dataclass
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from drn import GLM, CANN, MDN, DDR, DRN, preprocess_data, drn_cutpoints, train

from generate_synthetic_dataset import generate_synthetic_gamma

from analysis_utils import (
    generate_latex_table_more_runs,
    plot_metrics_grid,
    calculate_metrics,
)

torch.set_num_threads(1)

In [ ]:
TABLE_DIR = Path("tables/baseline-test/")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = Path("plots/baseline-test/")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Create a shared large test set (without standardising - it will need to be adjusted for each model later).
x_test_raw_shared, y_test_raw_shared, _, _ = generate_synthetic_gamma(
    10_000, seed=131313
)
x_test_raw_shared = pd.DataFrame(x_test_raw_shared, columns=["X_1", "X_2"])
Y_test_shared = torch.Tensor(y_test_raw_shared.values).flatten()

In [ ]:
np.random.seed(32052025)

NUM_DATASET_SEEDS = 20
dataset_seeds = [
    int(s) for s in np.random.randint(0, 2**32 - 1, size=NUM_DATASET_SEEDS)
]

sizes = [1_000, 3_000, 6_000]
datasets = {}

for i, dataset_seed in enumerate(dataset_seeds):
    # For each seed, make the largest dataset size
    features, target, _, _ = generate_synthetic_gamma(
        int(sizes[-1] * 0.8), seed=dataset_seed
    )

    x_train_raw, x_val_raw, y_train_raw, y_val_raw = train_test_split(
        features, target, test_size=0.25, random_state=42, shuffle=True
    )

    # Then subset it to the smaller sizes
    for size in sizes:
        x_train_size = x_train_raw[: int(0.6 * size)]
        y_train = y_train_raw[: int(0.6 * size)]
        x_val_size = x_val_raw[: int(0.2 * size)]
        y_val = y_val_raw[: int(0.2 * size)]

        x_train, x_val, x_test_shared, _, _ = preprocess_data(
            x_train_size,
            x_val_size,
            x_test_raw_shared,
            num_features=["X_1", "X_2"],
            cat_features=[],
            num_standard=True,
        )

        datasets[(size, i)] = (x_train, y_train, x_val, y_val, x_test_shared)

# Table D.6

In [ ]:
@dataclass
class TrainParams:
    proportion: float
    hidden_size: int
    dropout_rate: float
    num_hidden_layers: int
    lr: float
    batch_size: int
    patience: int
    kl_alpha: float


SIZE_TO_PARAMS: dict[int, TrainParams] = {
    1000: TrainParams(
        proportion=0.2,
        hidden_size=128,
        dropout_rate=0.5,
        num_hidden_layers=2,
        lr=1e-3 / 4,
        batch_size=128,
        patience=100,
        kl_alpha=0.05,
    ),
    3000: TrainParams(
        proportion=0.1,
        hidden_size=256,
        dropout_rate=0.4,
        num_hidden_layers=2,
        lr=1e-3 / 4,
        batch_size=128,
        patience=100,
        kl_alpha=0.05,
    ),
    6000: TrainParams(
        proportion=0.1,
        hidden_size=512,
        dropout_rate=0.3,
        num_hidden_layers=2,
        lr=1e-3 / 4,
        batch_size=128,
        patience=100,
        kl_alpha=0.05,
    ),
}

distribution = "gamma"

In [ ]:
results_batches = []

for size in [1_000, 3_000, 6_000]:
    for i in range(NUM_DATASET_SEEDS):
        x_train, y_train, x_val, y_val, x_test_shared = datasets[(size, i)]
        X_train = torch.Tensor(x_train.values)
        X_val = torch.Tensor(x_val.values)
        X_test_shared = torch.Tensor(x_test_shared.values)
        Y_train = torch.Tensor(y_train.values).flatten()
        Y_val = torch.Tensor(y_val.values).flatten()

        glm_gamma = GLM.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = GLM.from_statsmodels(X_train, Y_train, distribution="inversegaussian")
        glm_ig.eval()

        glm_gamma_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="gamma", null_model=True
        )
        glm_ig_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian", null_model=True
        )

        glm_gamma_empty = GLM(p=2, distribution="gamma")
        glm_ig_empty = GLM(p=2, distribution="inversegaussian")

        results_batch = calculate_metrics(
            models=[
                glm_ig_empty,
                glm_gamma_empty,
                glm_ig_null,
                glm_gamma_null,
                glm_ig,
                glm_gamma,
            ],
            names=[
                "GLM_IG_BIASED",
                "GLM_GA_BIASED",
                "GLM_IG_NULL",
                "GLM_GA_NULL",
                "GLM_IG",
                "GLM_GA",
            ],
            X_test_data=X_test_shared,
            Y_test_data=Y_test_shared,
            y_train=y_train,
            train_size=X_train.shape[0],
            seed_index=i,
        )
        results_batches.append(results_batch)


results = pd.concat(results_batches)
results.to_csv(TABLE_DIR / "baseline_glms.csv", index=False)

plot_metrics_grid(results)
plt.savefig(PLOT_DIR / "baseline_glms.png", dpi=300, bbox_inches="tight")

latex_code = generate_latex_table_more_runs(results)
with open(TABLE_DIR / "baseline_glms.tex", "w") as f:
    f.write(latex_code)

# Top Panel of Table D.7

In [ ]:
results_batches = []

for size in [1000, 3000, 6000]:
    params = SIZE_TO_PARAMS[size]
    proportion = params.proportion
    hidden_size = params.hidden_size
    dropout_rate = params.dropout_rate
    num_hidden_layers = params.num_hidden_layers
    lr = params.lr
    batch_size = params.batch_size
    patience = params.patience
    kl_alpha = params.kl_alpha

    for i in range(NUM_DATASET_SEEDS):
        x_train, y_train, x_val, y_val, x_test_shared = datasets[(size, i)]
        X_train = torch.Tensor(x_train.values)
        X_val = torch.Tensor(x_val.values)
        X_test_shared = torch.Tensor(x_test_shared.values)
        Y_train = torch.Tensor(y_train.values).flatten()
        Y_val = torch.Tensor(y_val.values).flatten()

        train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
        val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

        print(f"Size: {size}; Seed: {i}")
        print(
            "-----------------------------------------------------------------------------------"
        )

        glm_gamma = GLM.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = GLM.from_statsmodels(X_train, Y_train, distribution="inversegaussian")
        glm_ig.eval()

        cutpoints_DRN = drn_cutpoints(
            c_0=(
                np.min(Y_train.detach().numpy()) * 1.1
                if np.min(Y_train.detach().numpy()) < 0
                else 0.0
            ),
            c_K=20,
            proportion=proportion,
            y=Y_train.detach().numpy(),
            min_obs=3,
        )

        torch.manual_seed(23)
        drn_gamma = DRN(
            baseline=glm_gamma,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="forwards",
        )

        train(
            model=drn_gamma,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=50,
        )
        drn_gamma.eval()

        torch.manual_seed(23)
        drn_ig = DRN(
            baseline=glm_ig,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="forwards",
        )

        train(
            model=drn_ig,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_ig.eval()

        torch.manual_seed(23)
        drn_ig_small_kl = DRN(
            baseline=glm_ig,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="forwards",
        )

        train(
            model=drn_ig_small_kl,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_ig_small_kl.eval()

        torch.manual_seed(23)
        ddr = DDR(
            cutpoints_DRN,
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
        )
        train(
            ddr,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            lr=lr,
            batch_size=batch_size,
            log_interval=100,
            patience=patience,
            epochs=5000,
        )
        ddr.eval()

        torch.manual_seed(23)
        mdn = MDN(
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
            num_components=5,
            distribution=distribution,
        )

        torch.manual_seed(23)
        train(
            mdn,
            train_dataset,
            val_dataset,
            lr=lr,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            log_interval=100,
        )
        mdn.eval()

        results_batch = calculate_metrics(
            models=[drn_ig, drn_ig_small_kl, ddr, mdn, drn_gamma],
            names=["DRN_IG", "DRN_IG_KL_SMALL", "DDR", "MDN", "DRN_GA"],
            X_test_data=X_test_shared,
            Y_test_data=Y_test_shared,
            y_train=y_train,
            train_size=X_train.shape[0],
            seed_index=i,
        )
        results_batches.append(results_batch)

results = pd.concat(results_batches)
results.to_csv(TABLE_DIR / f"drn_vs_ddr_{NUM_DATASET_SEEDS}_seeds.csv", index=False)

plot_metrics_grid(results)
plt.savefig(PLOT_DIR / f"drn_vs_ddr_{NUM_DATASET_SEEDS}_seeds.png")

latex_code = generate_latex_table_more_runs(results)
with open(TABLE_DIR / f"drn_vs_ddr_{NUM_DATASET_SEEDS}_seeds.tex", "w") as f:
    f.write(latex_code)

# Bottom Panel of Table D.7

In [ ]:
results_batches = []

for size in [1000, 3000, 6000]:
    # PL NB: size=1000 Proportion was *2, and in all kl_alpha was 0.01
    params = SIZE_TO_PARAMS[size]
    proportion = params.proportion
    hidden_size = params.hidden_size
    dropout_rate = params.dropout_rate
    num_hidden_layers = params.num_hidden_layers
    lr = params.lr
    batch_size = params.batch_size
    patience = params.patience
    kl_alpha = params.kl_alpha

    for i in range(NUM_DATASET_SEEDS):
        x_train, y_train, x_val, y_val, x_test_shared = datasets[(size, i)]
        X_train = torch.Tensor(x_train.values)
        X_val = torch.Tensor(x_val.values)
        X_test_shared = torch.Tensor(x_test_shared.values)
        Y_train = torch.Tensor(y_train.values).flatten()
        Y_val = torch.Tensor(y_val.values).flatten()

        train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
        val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

        print(f"Size: {size}; Seed: {i}")
        print(
            "-----------------------------------------------------------------------------------"
        )

        glm_gamma = GLM.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = GLM.from_statsmodels(X_train, Y_train, distribution="inversegaussian")
        glm_ig.eval()

        glm_gamma_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="gamma", null_model=True
        )
        glm_ig_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian", null_model=True
        )

        glm_gamma_empty = GLM(p=2, distribution="gamma")
        glm_ig_empty = GLM(p=2, distribution="inversegaussian")

        cutpoints_DRN = drn_cutpoints(
            c_0=(
                np.min(Y_train.detach().numpy()) * 1.1
                if np.min(Y_train.detach().numpy()) < 0
                else 0.0
            ),
            c_K=20,  # np.max(Y_train.detach().numpy()) * 1.1,
            proportion=proportion,
            y=Y_train.detach().numpy(),
            min_obs=3,
        )

        torch.manual_seed(23)
        drn_gamma_null = DRN(
            baseline=glm_gamma_null,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="backward",
        )

        train(
            model=drn_gamma_null,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_gamma_null.eval()

        torch.manual_seed(23)
        drn_gamma_empty = DRN(
            baseline=glm_gamma_empty,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="backward",
        )
        train(
            model=drn_gamma_empty,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_gamma_empty.eval()

        torch.manual_seed(23)
        cann_gamma_null = CANN(
            glm_gamma_null,
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
        )

        train(
            cann_gamma_null,
            train_dataset,
            val_dataset,
            epochs=5000,
            lr=lr,
            patience=patience,
            batch_size=batch_size,
            log_interval=100,
        )
        cann_gamma_null.update_dispersion(X_train, Y_train)
        cann_gamma_null.eval()

        torch.manual_seed(23)
        cann_gamma_empty = CANN(
            glm_gamma_empty,
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
        )

        torch.manual_seed(23)
        train(
            cann_gamma_empty,
            train_dataset,
            val_dataset,
            epochs=5000,
            lr=lr,
            patience=patience,
            batch_size=batch_size,
            log_interval=100,
        )
        cann_gamma_empty.update_dispersion(X_train, Y_train)
        cann_gamma_empty.eval()

        results_batch = calculate_metrics(
            models=[drn_gamma_empty, drn_gamma_null, cann_gamma_empty, cann_gamma_null],
            names=["DRN_GA_EMPTY", "DRN_GA_NULL", "CANN_GA_EMPTY", "CANN_GA_NULL"],
            X_test_data=X_test_shared,
            Y_test_data=Y_test_shared,
            y_train=y_train,
            train_size=X_train.shape[0],
            seed_index=i,
        )
        results_batches.append(results_batch)

results = pd.concat(results_batches)
results.to_csv(TABLE_DIR / f"drn_vs_cann_{NUM_DATASET_SEEDS}_seeds.csv", index=False)

plot_metrics_grid(results)
plt.savefig(PLOT_DIR / f"drn_vs_cann_{NUM_DATASET_SEEDS}_seeds.png")

latex_code = generate_latex_table_more_runs(results)
with open(TABLE_DIR / f"drn_vs_cann_{NUM_DATASET_SEEDS}_seeds.tex", "w") as f:
    f.write(latex_code)